# DATA PREPARATION

This notebook prepares 5 datasets for model training and evaluation:

1. **LLM-annotated data**: Original LLM annotations from the full dataset
2. **Initial manual annotations**: First round of human-coded validation (500 articles)
3. **3 x Verified manual annotations**: Manually verified model predictions

All datasets use the CARDS taxonomy (categories 0–7), with:
- Multi-label encoding: one-hot columns `label_0` … `label_7`
- Binary encoding: `has_claim` (0 = no claim, 1 = contains any claim from categories 1–7)

## imports and configuration

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
# file paths 
RAW_ARTICLES  = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/raw/climate_misinfo_2023-2025.csv"
LLM_ANNOT     = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/llm_18k_annotated.parquet"
MANUAL_ANNOT_500 = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/manual_500_annotated.xlsx"
MANUAL_ANNOT_2000 = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/manual_2000_annotated.xlsx"
MANUAL_ANNOT_121 = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/manual_121_annotated.xlsx"
MANUAL_ANNOT_182 = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/annotated/manual_182_annotated.xlsx"

# output paths
OUT_LLM = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/cycle1/cleaned/llm_18k_clean.parquet"
OUT_MANUAL_500 = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/test_manual_500.parquet"
OUT_MANUAL_2000 = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/cycle2/cleaned/manual_2000_clean.parquet"
OUT_MANUAL_1921 = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/cycle3/cleaned/manual_1921_clean.parquet"
OUT_MANUAL_2103 = "/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/cycle4/cleaned/manual_2103_clean.parquet"

# label columns 
LABEL_COLS = [f"label_{i}" for i in range(8)]

## llm annotated data

In [3]:
# Load raw articles and LLM annotations
articles = pd.read_csv(RAW_ARTICLES)
llm_annot = pd.read_parquet(LLM_ANNOT)

print(f"Loaded {len(articles):,} articles")
print(f"Loaded {len(llm_annot):,} LLM annotations")
llm_annot.head(2)

Loaded 182,584 articles
Loaded 17,976 LLM annotations


,ArticleKey,categories,prompt_tokens,completion_tokens
0,ea4d58f9,[1_7_0],4401,41
1,ead59516,[1_7_0],4570,41


In [4]:
# rename and extract top-level CARDS category from subclaim codes
# subclaim entries look like "1_3_4"; the first digit is the top-level category
llm_annot.rename(columns={"categories": "CardsSubClaim"}, inplace=True)

llm_annot["CardsClaim"] = llm_annot["CardsSubClaim"].apply(
    lambda arr: sorted({str(x).split("_")[0] for x in arr})
    if isinstance(arr, (list, np.ndarray)) else []
)

# Drop token count columns (not needed for training)
llm_annot.drop(columns=["completion_tokens", "prompt_tokens"], inplace=True, errors="ignore")

In [5]:
# merge annotations with article metadata
llm_df = articles.merge(llm_annot, on="ArticleKey")
print(f"Merged dataset: {len(llm_df):,} articles")

Merged dataset: 17,976 articles


In [6]:
# validate labels and check distribution
unique_labels = sorted({label for labels in llm_df["CardsClaim"] for label in labels})
label_counts = Counter(label for labels in llm_df["CardsClaim"] for label in labels)

print("Unique labels found:", unique_labels)
print("\nLabel counts:")
for label in sorted(label_counts.keys(), key=lambda x: int(x) if x.isdigit() else 999):
    print(f"  {label}: {label_counts[label]:,}")

Unique labels found: ['0', '1', '12', '2', '3', '4', '5', '6', '7']

Label counts:
  0: 911
  1: 2,827
  2: 473
  3: 2,871
  4: 14,751
  5: 337
  6: 536
  7: 789
  12: 1


In [7]:
# examine occurences of label 12
llm_df[llm_df["CardsClaim"].apply(lambda x: "12" in x)]

,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,Medietype,Date,Month,Year,NewText,GeneralSentiment,SentimentLabel,Ritzau-telegram,CardsSubClaim,CardsClaim
10747,ea5abac3,2024-08-18 00:00:00,"Føler du, at verden er af lave? Her er 19 gode...",Jyllands-Posten har ringet til 15 danske ekspe...,1DE GLOBALE CO2-UDLEDNINGER TOPPER MÅSKE SNART...,1319,da,LASSE MOMME - BENJAMIN JUNGDAL THORGERD JEN...,Jyllands-Posten,JYP,...,Landsdækkende dagblade,2024-08-18,8,2024,"Føler du, at verden er af lave? Her er 19 gode...",Neutral,0.0,False,"[12_0_0, 4_2_0]","[12, 4]"


In [8]:
# remove invalid label "12" from any article that contains it.
llm_df["CardsClaim"] = llm_df["CardsClaim"].apply(lambda labels: [l for l in labels if l != "12"])

In [9]:
# convert string labels to integers and create one-hot encoding
llm_df["CardsClaim"] = llm_df["CardsClaim"].apply(lambda labels: [int(l) for l in labels])

# fit MultiLabelBinarizer on all 8 classes
mlb = MultiLabelBinarizer(classes=range(8))
llm_df[LABEL_COLS] = mlb.fit_transform(llm_df["CardsClaim"])

# Create binary label: 0 if no claim (only label_0), 1 if any claim (labels 1-7)
llm_df["has_claim"] = (llm_df[LABEL_COLS[1:]].sum(axis=1) > 0).astype(int)

print(f"\nLLM data prepared: {len(llm_df):,} articles")
print(f"\nBinary distribution:")
print(f"  No claim (0): {(llm_df['has_claim'] == 0).sum():5,} ({100 * (llm_df['has_claim'] == 0).sum() / len(llm_df):5.2f}%)")
print(f"  Has claim (1): {(llm_df['has_claim'] == 1).sum():5,} ({100 * (llm_df['has_claim'] == 1).sum() / len(llm_df):5.2f}%)")
print("\nMulti-label distribution:")
for i, col in enumerate(LABEL_COLS):
    count = llm_df[col].sum()
    pct = 100 * count / len(llm_df)
    print(f"  {col}: {count:5,} ({pct:5.2f}%)")


LLM data prepared: 17,976 articles

Binary distribution:
  No claim (0):   896 ( 4.98%)
  Has claim (1): 17,080 (95.02%)

Multi-label distribution:
  label_0:   911 ( 5.07%)
  label_1: 2,827 (15.73%)
  label_2:   473 ( 2.63%)
  label_3: 2,871 (15.97%)
  label_4: 14,751 (82.06%)
  label_5:   337 ( 1.87%)
  label_6:   536 ( 2.98%)
  label_7:   789 ( 4.39%)


In [13]:
# Save processed LLM data
llm_df.to_parquet(OUT_LLM, index=False)
print(f"\nSaved to {OUT_LLM}")


Saved to /Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/llm_data_clean.parquet


## initial manual annotations

This section processes the first round of manually coded validation data (500 articles).

In [14]:
# load manual annotations
manual_initial_df = pd.read_excel(MANUAL_ANNOT_500)
manual_initial_df.rename(columns={"ClaimCat": "HumanClaim", "SubClaimCat": "HumanSubClaim"}, inplace=True)

print(f"Loaded {len(manual_initial_df):,} manually annotated articles (initial round)")
manual_initial_df.head(2)

Loaded 500 manually annotated articles (initial round)


,ArticleKey,DateTime,Headline,Subheadline,ArticleText,HumanClaim,HumanSubClaim,wordcountbodytext,language,byline,...,Readership,Link,Medietype,Date,Month,Year,NewText,GeneralSentiment,SentimentLabel,Ritzau-telegram
0,eb0e1b6e,2025-11-18 23:21:33,Ukraine forbereder krav til Rusland for miljøs...,NaN,Krig har skabt ekstra udledninger svarende til...,0_0_0,0_0_0,282,da,NaN,...,1250,https://ditikast-brande.dk/artikel/39577688-e4...,Webkilder,2025-11-18,11,2025,Ukraine forbereder krav til Rusland for miljøs...,Negative,-1.0,True
1,eac48880,2025-05-28 08:31:54,Ny rapport: EU-lande er godt på vej til at ind...,NaN,Udland Ny rapport: EU-lande er godt på vej til...,0_0_0,0_0_0,479,da,Ritzau,...,114140,https://stiften.dk/udland/ny-rapport-eu-er-god...,Webkilder,2025-05-28,5,2025,Ny rapport: EU-lande er godt på vej til at ind...,Positive,1.0,True


In [15]:
# parse human claim labels
# HumanClaim is stored as semicolon-separated hierarchical codes, e.g., "3_0_0;0_0_0"
# extract only the top-level category (first digit)
def parse_human_claims(cell):
    if pd.isna(cell):
        return [0]
    claims = str(cell).split(";")
    return sorted({int(c.split("_")[0]) for c in claims if c.strip()})

manual_initial_df["HumanClaim"] = manual_initial_df["HumanClaim"].apply(parse_human_claims)

# Show label distribution
label_counts_manual = Counter(label for labels in manual_initial_df["HumanClaim"] for label in labels)
print("\nInitial manual annotation label counts:")
for label in sorted(label_counts_manual.keys()):
    print(f"  {label}: {label_counts_manual[label]}")


Initial manual annotation label counts:
  0: 451
  1: 1
  2: 1
  3: 1
  4: 41
  5: 4
  6: 6
  7: 5


In [16]:
# apply the same MultiLabelBinarizer to ensure consistent encoding
manual_initial_df[LABEL_COLS] = mlb.transform(manual_initial_df["HumanClaim"])

# create binary label: 0 if no claim (only label_0), 1 if any claim (labels 1-7)
manual_initial_df["has_claim"] = (manual_initial_df[LABEL_COLS[1:]].sum(axis=1) > 0).astype(int)

print(f"\nInitial manual data prepared: {len(manual_initial_df):,} articles")
print(f"\nBinary distribution:")
print(f"  No claim (0): {(manual_initial_df['has_claim'] == 0).sum():3,} ({100 * (manual_initial_df['has_claim'] == 0).sum() / len(manual_initial_df):5.2f}%)")
print(f"  Has claim (1): {(manual_initial_df['has_claim'] == 1).sum():3,} ({100 * (manual_initial_df['has_claim'] == 1).sum() / len(manual_initial_df):5.2f}%)")
print("\nMulti-label distribution:")
for i, col in enumerate(LABEL_COLS):
    count = manual_initial_df[col].sum()
    pct = 100 * count / len(manual_initial_df)
    print(f"  {col}: {count:3,} ({pct:5.2f}%)")


Initial manual data prepared: 500 articles

Binary distribution:
  No claim (0): 451 (90.20%)
  Has claim (1):  49 ( 9.80%)

Multi-label distribution:
  label_0: 451 (90.20%)
  label_1:   1 ( 0.20%)
  label_2:   1 ( 0.20%)
  label_3:   1 ( 0.20%)
  label_4:  41 ( 8.20%)
  label_5:   4 ( 0.80%)
  label_6:   6 ( 1.20%)
  label_7:   5 ( 1.00%)


In [19]:
# save processed initial manual data
manual_initial_df.to_parquet(OUT_MANUAL_INITIAL, index=False)
print(f"\nSaved to {OUT_MANUAL_INITIAL}")


Saved to /Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/manual_initial_clean.parquet


## verified manual annotations - cycle 1

This section processes the manually verified predictions from the validation set (2,000 articles).  
These observations consists of the observations in the llm validation set and is extracted from the models predictions and then manually reviewed and corrected

In [ ]:
# load verified predictions
manual_verified_df = pd.read_excel(MANUAL_ANNOT_2000, sheet_name="annotated")

print(f"Loaded {len(manual_verified_df):,} verified predictions")
print(f"\nColumns in verified data: {manual_verified_df.columns.tolist()}")
manual_verified_df.head(2)

Loaded 2,002 verified predictions

Columns in verified data: ['ArticleKey', 'NewsText', 'VerifiedClaim', 'BinaryLabel']


,ArticleKey,NewsText,VerifiedClaim,BinaryLabel
0,eb2c4dc2,Elbiler bør ikke længere være undtaget fra at ...,0_0_0,0
1,ea339964,Mennesket har opbygget en fantastisk civilisat...,0_0_0,0


In [21]:
# configure column names
# the column containing the verified labels
VERIFIED_LABEL_COL = "VerifiedClaim"

# the column containing the article identifier
ARTICLE_KEY_COL = "ArticleKey" 

print(f"Using label column: '{VERIFIED_LABEL_COL}'")
print(f"Using article key column: '{ARTICLE_KEY_COL}'")

Using label column: 'VerifiedClaim'
Using article key column: 'ArticleKey'


In [22]:
# merge annotations with article metadata
manual_verified_df = articles.merge(manual_verified_df, on="ArticleKey")
print(f"Merged dataset: {len(manual_verified_df):,} articles")

Merged dataset: 2,002 articles


In [27]:
# drop extra NewsText-column
column_to_drop = ["NewsText"]

manual_verified_df.drop(columns=column_to_drop, inplace=True)

manual_verified_df.head(2)

,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,Medietype,Date,Month,Year,NewText,GeneralSentiment,SentimentLabel,Ritzau-telegram,VerifiedClaim,BinaryLabel
0,e996a478,2023-05-03 07:14:35,KLIMA: C25-selskaber mangler ESG-kompetencer,NaN,"C25-bestyrelser mangler ESG-kompetencer, skriv...",90,da,NaN,24nyt.dk,3BI,...,Webkilder,2023-05-03,5,2023,KLIMA: C25-selskaber mangler ESG-kompetencer C...,Neutral,0.0,False,0_0_0,0
1,e9fdf866,2023-11-29 22:54:26,DF forlader forhandlinger: Plan om kæmpevindmø...,NaN,Dansk Folkeparti har onsdag eftermiddag meddel...,329,da,NaN,24nyt.dk,3BI,...,Webkilder,2023-11-29,11,2023,DF forlader forhandlinger: Plan om kæmpevindmø...,Negative,-1.0,False,0_0_0,0


In [ ]:
# process verified labels
# VerifiedClaim format: "4_0_0" or "7_0_0; 4_0_0" 
# extract first digit from each subclaim to get the top-level category

def parse_verified_claims(cell):
    """Parse verified claim labels.
    
    Format: "4_0_0" or "7_0_0; 4_0_0"
    Extracts first digit from each subclaim code.
    Label 0 is mutually exclusive with other labels.
    """
    cell = str(cell).strip()
    
    # split by semicolon and extract first digit from each code
    claims = cell.split(";")
    labels = sorted({int(c.strip().split("_")[0]) for c in claims})
    
    # if label 0 appears with others, keep only label 0
    if 0 in labels and len(labels) > 1:
        return [0]
    
    return labels

manual_verified_df["VerifiedClaim"] = manual_verified_df[VERIFIED_LABEL_COL].apply(parse_verified_claims)

# show label distribution
label_counts_verified = Counter(label for labels in manual_verified_df["VerifiedClaim"] for label in labels)
print("\nVerified annotation label counts:")
for label in sorted(label_counts_verified.keys()):
    count = label_counts_verified[label]
    pct = 100 * count / len(manual_verified_df)
    print(f"  {label}: {count:4,} ({pct:5.2f}%)")


Verified annotation label counts:
  0: 1,901 (94.96%)
  1:    2 ( 0.10%)
  2:    2 ( 0.10%)
  3:    3 ( 0.15%)
  4:   90 ( 4.50%)
  5:    3 ( 0.15%)
  6:    9 ( 0.45%)
  7:    6 ( 0.30%)


In [29]:
# Apply the same MultiLabelBinarizer to ensure consistent encoding
manual_verified_df[LABEL_COLS] = mlb.transform(manual_verified_df["VerifiedClaim"])

# Create binary label: 0 if no claim (only label_0), 1 if any claim (labels 1-7)
manual_verified_df["has_claim"] = (manual_verified_df[LABEL_COLS[1:]].sum(axis=1) > 0).astype(int)

print(f"\nVerified manual data prepared: {len(manual_verified_df):,} articles")
print(f"\nBinary distribution:")
print(f"  No claim (0): {(manual_verified_df['has_claim'] == 0).sum():4,} ({100 * (manual_verified_df['has_claim'] == 0).sum() / len(manual_verified_df):5.2f}%)")
print(f"  Has claim (1): {(manual_verified_df['has_claim'] == 1).sum():4,} ({100 * (manual_verified_df['has_claim'] == 1).sum() / len(manual_verified_df):5.2f}%)")
print("\nMulti-label distribution:")
for i, col in enumerate(LABEL_COLS):
    count = manual_verified_df[col].sum()
    pct = 100 * count / len(manual_verified_df)
    print(f"  {col}: {count:4,} ({pct:5.2f}%)")


Verified manual data prepared: 2,002 articles

Binary distribution:
  No claim (0): 1,901 (94.96%)
  Has claim (1):  101 ( 5.04%)

Multi-label distribution:
  label_0: 1,901 (94.96%)
  label_1:    2 ( 0.10%)
  label_2:    2 ( 0.10%)
  label_3:    3 ( 0.15%)
  label_4:   90 ( 4.50%)
  label_5:    3 ( 0.15%)
  label_6:    9 ( 0.45%)
  label_7:    6 ( 0.30%)


In [30]:
# Save processed verified manual data
manual_verified_df.to_parquet(OUT_MANUAL_2000, index=False)
print(f"\nSaved to {OUT_MANUAL_2000}")


Saved to /Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/manual_verified_clean.parquet


## verified manual annotations - cycle 2

This section processes the manually verified predictions from the extra sample of 9K test set extracted from the raw dataset of 182K observationsin from the database. These observations has never been annotated by the llm or used for model training. Out of these 9K the model trained for binary classification predicted a positive label in 189 of the cases. Out of these 189 I verified that 121 of them is indeed a positive label. These observations are gonna be added to the training data for the binary model to obtain a more balanced data set

In [14]:
# load verified predictions
manual_verified_df = pd.read_excel(MANUAL_ANNOT_121, sheet_name="annotated")

print(f"Loaded {len(manual_verified_df):,} verified predictions")
print(f"\nColumns in verified data: {manual_verified_df.columns.tolist()}")
manual_verified_df.head(2)

Loaded 121 verified predictions

Columns in verified data: ['ArticleKey', 'VerifiedClaim']


,ArticleKey,VerifiedClaim
0,e96c02df,4_0_0
1,e96bb927,4_0_0


In [15]:
# configure column names
# the column containing the verified labels
VERIFIED_LABEL_COL = "VerifiedClaim"

# the column containing the article identifier
ARTICLE_KEY_COL = "ArticleKey" 

print(f"Using label column: '{VERIFIED_LABEL_COL}'")
print(f"Using article key column: '{ARTICLE_KEY_COL}'")

Using label column: 'VerifiedClaim'
Using article key column: 'ArticleKey'


In [21]:
# merge annotations with article metadata
manual_verified_df = articles.merge(manual_verified_df, on="ArticleKey")
print(f"Merged dataset: {len(manual_verified_df):,} articles")

manual_verified_df.head(2)

Merged dataset: 121 articles


,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,Medietype,Date,Month,Year,NewText,GeneralSentiment,SentimentLabel,Ritzau-telegram,VerifiedClaim,BinaryClaim
0,e97b02ed,2023-03-17 12:01:59,Regeringens nye afgift på godskørsel er gift f...,NaN,"Læserbrev: Regeringen, inklusive Venstre, der ...",470,da,Hans Kristian Skibby,Amtsavisen.dk (Randers Amtsavis),4E2,...,Webkilder,2023-03-17,3,2023,Regeringens nye afgift på godskørsel er gift f...,Neutral,0.0,False,4_0_0,1
1,e9e3ef52,2023-09-27 00:00:00,Debat:LÆSERNE MENER Hold endelig vindmøller la...,NaN,"Folkekirken har en lovfæstet ret til at nægte,...",308,da,NaN,Berlingske,BMA,...,Landsdækkende dagblade,2023-09-27,9,2023,Debat:LÆSERNE MENER Hold endelig vindmøller la...,Positive,1.0,False,4_0_0,1


In [22]:
# process verified labels
# VerifiedClaim format: "4_0_0" or "7_0_0; 4_0_0" 
# extract first digit from each subclaim to get the top-level category

def parse_verified_claims(cell):
    """Parse verified claim labels.
    
    Format: "4_0_0" or "7_0_0; 4_0_0"
    Extracts first digit from each subclaim code.
    Label 0 is mutually exclusive with other labels.
    """
    cell = str(cell).strip()
    
    # split by semicolon and extract first digit from each code
    claims = cell.split(";")
    labels = sorted({int(c.strip().split("_")[0]) for c in claims})
    
    # if label 0 appears with others, keep only label 0
    if 0 in labels and len(labels) > 1:
        return [0]
    
    return labels

manual_verified_df["VerifiedClaim"] = manual_verified_df[VERIFIED_LABEL_COL].apply(parse_verified_claims)

# show label distribution
label_counts_verified = Counter(label for labels in manual_verified_df["VerifiedClaim"] for label in labels)
print("\nVerified annotation label counts:")
for label in sorted(label_counts_verified.keys()):
    count = label_counts_verified[label]
    pct = 100 * count / len(manual_verified_df)
    print(f"  {label}: {count:4,} ({pct:5.2f}%)")


Verified annotation label counts:
  1:    1 ( 0.83%)
  2:    5 ( 4.13%)
  3:   14 (11.57%)
  4:  111 (91.74%)
  5:    4 ( 3.31%)
  6:   16 (13.22%)
  7:    3 ( 2.48%)


In [23]:
# Apply the same MultiLabelBinarizer to ensure consistent encoding
manual_verified_df[LABEL_COLS] = mlb.transform(manual_verified_df["VerifiedClaim"])

# Create binary label: 0 if no claim (only label_0), 1 if any claim (labels 1-7)
manual_verified_df["has_claim"] = (manual_verified_df[LABEL_COLS[1:]].sum(axis=1) > 0).astype(int)

print(f"\nVerified manual data prepared: {len(manual_verified_df):,} articles")
print(f"\nBinary distribution:")
print(f"  No claim (0): {(manual_verified_df['has_claim'] == 0).sum():4,} ({100 * (manual_verified_df['has_claim'] == 0).sum() / len(manual_verified_df):5.2f}%)")
print(f"  Has claim (1): {(manual_verified_df['has_claim'] == 1).sum():4,} ({100 * (manual_verified_df['has_claim'] == 1).sum() / len(manual_verified_df):5.2f}%)")
print("\nMulti-label distribution:")
for i, col in enumerate(LABEL_COLS):
    count = manual_verified_df[col].sum()
    pct = 100 * count / len(manual_verified_df)
    print(f"  {col}: {count:4,} ({pct:5.2f}%)")

manual_verified_df.head(2)


Verified manual data prepared: 121 articles

Binary distribution:
  No claim (0):    0 ( 0.00%)
  Has claim (1):  121 (100.00%)

Multi-label distribution:
  label_0:    0 ( 0.00%)
  label_1:    1 ( 0.83%)
  label_2:    5 ( 4.13%)
  label_3:   14 (11.57%)
  label_4:  111 (91.74%)
  label_5:    4 ( 3.31%)
  label_6:   16 (13.22%)
  label_7:    3 ( 2.48%)


,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,BinaryClaim,label_0,label_1,label_2,label_3,label_4,label_5,label_6,label_7,has_claim
0,e97b02ed,2023-03-17 12:01:59,Regeringens nye afgift på godskørsel er gift f...,NaN,"Læserbrev: Regeringen, inklusive Venstre, der ...",470,da,Hans Kristian Skibby,Amtsavisen.dk (Randers Amtsavis),4E2,...,1,0,0,0,0,1,0,0,0,1
1,e9e3ef52,2023-09-27 00:00:00,Debat:LÆSERNE MENER Hold endelig vindmøller la...,NaN,"Folkekirken har en lovfæstet ret til at nægte,...",308,da,NaN,Berlingske,BMA,...,1,0,0,0,0,1,0,0,0,1


In [24]:
# Merge with the exisiting training data 
train_df = pd.read_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/train_manual_verified.parquet")
train_df.head(2)

,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,BinaryLabel,label_0,label_1,label_2,label_3,label_4,label_5,label_6,label_7,has_claim
0,e996a478,2023-05-03 07:14:35,KLIMA: C25-selskaber mangler ESG-kompetencer,None,"C25-bestyrelser mangler ESG-kompetencer, skriv...",90,da,None,24nyt.dk,3BI,...,0,1,0,0,0,0,0,0,0,0
1,e9fdf866,2023-11-29 22:54:26,DF forlader forhandlinger: Plan om kæmpevindmø...,None,Dansk Folkeparti har onsdag eftermiddag meddel...,329,da,None,24nyt.dk,3BI,...,0,1,0,0,0,0,0,0,0,0


In [25]:
# Concatenate the dataframes
merged_df = pd.concat([train_df, manual_verified_df], ignore_index=True)

print(f"\nMerged shape: {merged_df.shape}")
print(f"Original train_df: {len(train_df):,} articles")
print(f"New manual_verified_df: {len(manual_verified_df):,} articles")
print(f"Merged total: {len(merged_df):,} articles")


Merged shape: (1921, 33)
Original train_df: 1,800 articles
New manual_verified_df: 121 articles
Merged total: 1,921 articles


In [27]:
merged_df = merged_df.drop(columns="BinaryClaim")
merged_df.head(2)

,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,BinaryLabel,label_0,label_1,label_2,label_3,label_4,label_5,label_6,label_7,has_claim
0,e996a478,2023-05-03 07:14:35,KLIMA: C25-selskaber mangler ESG-kompetencer,None,"C25-bestyrelser mangler ESG-kompetencer, skriv...",90,da,None,24nyt.dk,3BI,...,0.0,1,0,0,0,0,0,0,0,0
1,e9fdf866,2023-11-29 22:54:26,DF forlader forhandlinger: Plan om kæmpevindmø...,None,Dansk Folkeparti har onsdag eftermiddag meddel...,329,da,None,24nyt.dk,3BI,...,0.0,1,0,0,0,0,0,0,0,0


In [28]:
# Save processed verified manual data
merged_df.to_parquet(OUT_MANUAL_121, index=False)
print(f"\nSaved to {OUT_MANUAL_121}")


Saved to /Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/interim/manual_verified_2_clean.parquet


## manual annotated - cycle 3

In [17]:
# load verified predictions
manual_182_df = pd.read_excel(MANUAL_ANNOT_182, sheet_name="annotated")

print(f"Loaded {len(manual_182_df):,} verified predictions")
print(f"\nColumns in verified data: {manual_182_df.columns.tolist()}")
manual_182_df.head(2)

Loaded 182 verified predictions

Columns in verified data: ['ArticleKey', 'VerifiedClaim']


,ArticleKey,VerifiedClaim
0,ea70bf7a,4_0_0
1,e97abe74,4_0_0


In [18]:
# configure column names
# the column containing the verified labels
VERIFIED_LABEL_COL = "VerifiedClaim"

# the column containing the article identifier
ARTICLE_KEY_COL = "ArticleKey" 

print(f"Using label column: '{VERIFIED_LABEL_COL}'")
print(f"Using article key column: '{ARTICLE_KEY_COL}'")

Using label column: 'VerifiedClaim'
Using article key column: 'ArticleKey'


In [19]:
# merge annotations with article metadata
manual_182_df = articles.merge(manual_182_df, on="ArticleKey")
print(f"Merged dataset: {len(manual_182_df):,} articles")

manual_182_df.head(2)

Merged dataset: 182 articles


,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,Link,Medietype,Date,Month,Year,NewText,GeneralSentiment,SentimentLabel,Ritzau-telegram,VerifiedClaim
0,e99decaf,2023-05-16 04:05:46,"Med lastbiloprøret skete præcis det, statsmini...",NaN,Statsministeren førte fra starten klimapolitik...,760,da,Daniel Bue Lauritzen Redaktør,Altinget.dk,52C,...,https://www.altinget.dk/transport/artikel/med-...,Webkilder,2023-05-16,5,2023,"Med lastbiloprøret skete præcis det, statsmini...",Negative,-1.0,False,6_0_0
1,e98f8f26,2023-04-21 11:25:54,Havet og jorden kan kun bruges én gang,NaN,"Læserbrev: Det er super vigtigt, at vi får sat...",435,da,Asger Christensen,Amtsavisen.dk (Randers Amtsavis),4E2,...,https://amtsavisen.dk/debat/havet-og-jorden-ka...,Webkilder,2023-04-21,4,2023,Havet og jorden kan kun bruges én gang Læserbr...,Positive,1.0,False,4_0_0


In [20]:
# process verified labels
# VerifiedClaim format: "4_0_0" or "7_0_0; 4_0_0" 
# extract first digit from each subclaim to get the top-level category

def parse_verified_claims(cell):
    """Parse verified claim labels.
    
    Format: "4_0_0" or "7_0_0; 4_0_0"
    Extracts first digit from each subclaim code.
    Label 0 is mutually exclusive with other labels.
    """
    cell = str(cell).strip()
    
    # split by semicolon and extract first digit from each code
    claims = cell.split(";")
    labels = sorted({int(c.strip().split("_")[0]) for c in claims})
    
    # if label 0 appears with others, keep only label 0
    if 0 in labels and len(labels) > 1:
        return [0]
    
    return labels

manual_182_df["VerifiedClaim"] = manual_182_df[VERIFIED_LABEL_COL].apply(parse_verified_claims)

# show label distribution
label_counts_verified = Counter(label for labels in manual_182_df["VerifiedClaim"] for label in labels)
print("\nVerified annotation label counts:")
for label in sorted(label_counts_verified.keys()):
    count = label_counts_verified[label]
    pct = 100 * count / len(manual_182_df)
    print(f"  {label}: {count:4,} ({pct:5.2f}%)")


Verified annotation label counts:
  2:    9 ( 4.95%)
  3:    6 ( 3.30%)
  4:  176 (96.70%)
  5:    9 ( 4.95%)
  6:   22 (12.09%)
  7:    3 ( 1.65%)


In [21]:
# Apply the same MultiLabelBinarizer to ensure consistent encoding
manual_182_df[LABEL_COLS] = mlb.transform(manual_182_df["VerifiedClaim"])

# Create binary label: 0 if no claim (only label_0), 1 if any claim (labels 1-7)
manual_182_df["has_claim"] = (manual_182_df[LABEL_COLS[1:]].sum(axis=1) > 0).astype(int)

print(f"\nVerified manual data prepared: {len(manual_182_df):,} articles")
print(f"\nBinary distribution:")
print(f"  No claim (0): {(manual_182_df['has_claim'] == 0).sum():4,} ({100 * (manual_182_df['has_claim'] == 0).sum() / len(manual_182_df):5.2f}%)")
print(f"  Has claim (1): {(manual_182_df['has_claim'] == 1).sum():4,} ({100 * (manual_182_df['has_claim'] == 1).sum() / len(manual_182_df):5.2f}%)")
print("\nMulti-label distribution:")
for i, col in enumerate(LABEL_COLS):
    count = manual_182_df[col].sum()
    pct = 100 * count / len(manual_182_df)
    print(f"  {col}: {count:4,} ({pct:5.2f}%)")

manual_182_df.head(2)


Verified manual data prepared: 182 articles

Binary distribution:
  No claim (0):    0 ( 0.00%)
  Has claim (1):  182 (100.00%)

Multi-label distribution:
  label_0:    0 ( 0.00%)
  label_1:    0 ( 0.00%)
  label_2:    9 ( 4.95%)
  label_3:    6 ( 3.30%)
  label_4:  176 (96.70%)
  label_5:    9 ( 4.95%)
  label_6:   22 (12.09%)
  label_7:    3 ( 1.65%)


,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,VerifiedClaim,label_0,label_1,label_2,label_3,label_4,label_5,label_6,label_7,has_claim
0,e99decaf,2023-05-16 04:05:46,"Med lastbiloprøret skete præcis det, statsmini...",NaN,Statsministeren førte fra starten klimapolitik...,760,da,Daniel Bue Lauritzen Redaktør,Altinget.dk,52C,...,[6],0,0,0,0,0,0,1,0,1
1,e98f8f26,2023-04-21 11:25:54,Havet og jorden kan kun bruges én gang,NaN,"Læserbrev: Det er super vigtigt, at vi får sat...",435,da,Asger Christensen,Amtsavisen.dk (Randers Amtsavis),4E2,...,[4],0,0,0,0,1,0,0,0,1


In [22]:
# Merge with the exisiting training data 
train_df = pd.read_parquet("/Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/cycle3/train_manual_1921.parquet")
train_df.head(2)

,ArticleKey,DateTime,Headline,Subheadline,ArticleText,wordcountbodytext,language,byline,Media,sourcecode,...,BinaryLabel,label_0,label_1,label_2,label_3,label_4,label_5,label_6,label_7,has_claim
0,e996a478,2023-05-03 07:14:35,KLIMA: C25-selskaber mangler ESG-kompetencer,None,"C25-bestyrelser mangler ESG-kompetencer, skriv...",90,da,None,24nyt.dk,3BI,...,0.0,1,0,0,0,0,0,0,0,0
1,e9fdf866,2023-11-29 22:54:26,DF forlader forhandlinger: Plan om kæmpevindmø...,None,Dansk Folkeparti har onsdag eftermiddag meddel...,329,da,None,24nyt.dk,3BI,...,0.0,1,0,0,0,0,0,0,0,0


In [23]:
# Concatenate the dataframes
merged_df = pd.concat([train_df, manual_182_df], ignore_index=True)

print(f"\nMerged shape: {merged_df.shape}")
print(f"Original train_df: {len(train_df):,} articles")
print(f"New manual_verified_df: {len(manual_182_df):,} articles")
print(f"Merged total: {len(merged_df):,} articles")


Merged shape: (2103, 32)
Original train_df: 1,921 articles
New manual_verified_df: 182 articles
Merged total: 2,103 articles


In [25]:
# Save processed verified manual data
merged_df.to_parquet(OUT_MANUAL_2103, index=False)
print(f"\nSaved to {OUT_MANUAL_2103}")


Saved to /Users/FrederikkeB/Documents/GitHub/misleading-climate-claims-dk/data/processed/cycle4/cleaned/manual_2103_clean.parquet
